In [ ]:
import pandas as pd
import lightgbm as lgb
import numpy as np

from sklearn.model_selection import train_test_split

In [ ]:
train_data = r'data\test.csv'

df = pd.read_csv(train_data, encoding='utf8')

In [ ]:
cat_cols = df.select_dtypes(include=['string']).columns

# 2. Convert them to pandas 'category' dtype
for col in cat_cols:
    df[col] = df[col].astype('category')

In [ ]:
df['SalePrice'] = np.log1p(df['SalePrice'])

X = df.drop(['SalePrice', 'Id'], axis=1)
y = df['SalePrice']

In [ ]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.25, random_state=42)

In [ ]:
train_data = lgb.Dataset(X_train, label=y_train, categorical_feature=list(cat_cols))
test_data = lgb.Dataset(X_val, label=y_val, reference=train_data)

In [ ]:
params = {
    'objective': 'regression',
    'metric': 'rmse',
    'boosting_type': 'gbdt',
    'num_leaves': 31,
    'learning_rate': 0.05,
    'feature_fraction': 0.9
}

In [ ]:
num_round=100
bst = lgb.train(params, train_data, num_round,valid_sets=[test_data], callbacks=[lgb.early_stopping(stopping_rounds=10)])

In [ ]:
bst.save_model('model.txt', num_iteration=bst.best_iteration)

In [ ]:
# import matplotlib.pyplot as plt


# fig, ax = plt.subplots(figsize=(12,12))
# lgb.plot_importance(bst, ax=ax)